# Experiment 2: Pause after the attention part, change its output, resume

Each of GPT-2's 12 blocks has two parts that run in order:
1. **Look-around part (attention):** every word gathers information from the words before it.
2. **Think part (MLP):** every word is processed on its own.

Each part **adds** its result to the running table of numbers.

In this notebook we:
1. Check that running each block in two parts gives exactly the normal answer.
2. **PAUSE** inside a block, right after its look-around part, and **save** the result.
3. Show what the last word was looking at.
4. **MODIFY** the saved look-around result.
5. **RESUME** with the think part of the same block, then the remaining blocks, **without rerunning** anything earlier.
6. Compare normal and modified predictions.

## Step 0: Install (Colab only)
Tested with PyTorch 2.14.1 and Transformers 5.20.2. Uncomment only if something breaks.

In [1]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.masking_utils import create_causal_mask

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer
NUM_BLOCKS = len(gpt2.h)

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)
print("Inside one block:", [name for name, _ in gpt2.h[0].named_children()])

/Users/arinpatel/Desktop/BTP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1023 00:13:09.846000 53861 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8761.57it/s]

Device: cpu
Number of blocks: 12
Inside one block: ['ln_1', 'attn', 'ln_2', 'mlp']


## Step 2: Normal run (no changes)

In [3]:
PROMPT = "I live in Paris, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
words = [tokenizer.decode(t) for t in input_ids[0]]
print("Word-pieces:", words)

FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

def top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

def compare(new_logits, title="Modified model"):
    """Print the normal and new top 5 side by side, plus the chance of ' French' and ' German'."""
    print(f"{'Normal model':<26}{title}")
    for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(new_logits)):
        print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")
    old, new = normal_logits[0, -1].softmax(-1), new_logits[0, -1].softmax(-1)
    print(f"\n  ' French': {old[FRENCH]:6.2%} -> {new[FRENCH]:6.2%}")
    print(f"  ' German': {old[GERMAN]:6.2%} -> {new[GERMAN]:6.2%}")

normal_logits = model(input_ids, use_cache=False).logits
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']



Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Run each block in two parts
- `look_part(i, state)` runs only the look-around part of block `i` and returns its result (one row of 768 numbers per word).
- `think_part(i, state, look_result)` adds the look-around result, then runs the think part and adds that too.
- `run_until_look(ids, layer)` runs everything before block `layer`, then that block's look-around part, then **PAUSES**.
  It returns a **checkpoint**: the table that entered the block, plus the look-around result.
- `resume_after_look(checkpoint)` **RESUMES** with the think part of the same block, then the remaining blocks.

In [4]:
def embed(input_ids):
    positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
    return gpt2.wte(input_ids) + gpt2.wpe(positions)

def mask_and_positions(state):
    positions = torch.arange(state.shape[1], device=state.device).unsqueeze(0)
    # rule that each word can only look at earlier words (same as the normal model)
    mask = create_causal_mask(config=model.config, inputs_embeds=state, attention_mask=None,
                              past_key_values=None, position_ids=positions)
    return mask, positions

def look_part(i, state):
    """Look-around (attention) part of block i. Returns its result; does NOT add it to the table."""
    block = gpt2.h[i]
    mask, positions = mask_and_positions(state)
    look_result, _ = block.attn(block.ln_1(state), attention_mask=mask, position_ids=positions)
    return look_result

def think_part(i, state, look_result):
    """Add the look-around result, then run the think (MLP) part of block i and add its result."""
    block = gpt2.h[i]
    state = state + look_result
    return state + block.mlp(block.ln_2(state))

def final_logits(state):
    return model.lm_head(gpt2.ln_f(state))

def run_model(input_ids, edits=None):
    """Full run, block by block, part by part. `edits` = {block number: function that changes its look-around result}."""
    edits = edits or {}
    state = embed(input_ids)
    for i in range(NUM_BLOCKS):
        look_result = look_part(i, state)
        if i in edits:
            look_result = edits[i](look_result)
        state = think_part(i, state, look_result)
    return final_logits(state)

def run_until_look(input_ids, layer):
    """Run blocks 0..layer-1 fully, then only the look-around part of block `layer`, then PAUSE."""
    state = embed(input_ids)
    for i in range(layer):
        state = think_part(i, state, look_part(i, state))
    return {"layer": layer, "block_input": state, "look_result": look_part(layer, state)}

def resume_after_look(checkpoint):
    """RESUME: think part of the paused block, then all later blocks."""
    layer = checkpoint["layer"]
    state = think_part(layer, checkpoint["block_input"], checkpoint["look_result"])
    for i in range(layer + 1, NUM_BLOCKS):
        state = think_part(i, state, look_part(i, state))
    return final_logits(state)

## Step 4: A counter that records which parts actually run
`look 5` means the look-around part of block 5 ran; `think 5` means the think part of block 5 ran.

In [5]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

parts_run = []
counter_hooks = []
for i, block in enumerate(gpt2.h):
    counter_hooks.append(block.attn.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"look {i}")))
    counter_hooks.append(block.mlp.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"think {i}")))
print("Counter attached to", len(counter_hooks), "parts (12 blocks x 2 parts)")

Counter attached to 24 parts (12 blocks x 2 parts)


## Step 5: Check that the two-part run matches the normal model

In [6]:
parts_run.clear()
split_logits = run_model(input_ids)

diff = (split_logits - normal_logits).abs().max().item()
print("Parts run:", parts_run)
print("Largest difference from normal model:", diff)
assert diff < 1e-4, "Two-part run does not match the normal model"
print("OK: two-part run matches the normal model")

Parts run: ['look 0', 'think 0', 'look 1', 'think 1', 'look 2', 'think 2', 'look 3', 'think 3', 'look 4', 'think 4', 'look 5', 'think 5', 'look 6', 'think 6', 'look 7', 'think 7', 'look 8', 'think 8', 'look 9', 'think 9', 'look 10', 'think 10', 'look 11', 'think 11']
Largest difference from normal model: 0.0
OK: two-part run matches the normal model


## Step 6: PAUSE after the look-around part of block 8
We pause in block 8 because that is the first block where the last word (' speak') looks strongly at ' Paris' (shown below).
The checkpoint holds two things:
- the table that **entered** block 8 (needed because the look-around result is *added* to it), and
- the look-around result of block 8.

In [7]:
PAUSE_LAYER = 8

parts_run.clear()
checkpoint = run_until_look(input_ids, PAUSE_LAYER)  # <-- PAUSE happens here
saved = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in checkpoint.items()}  # <-- SAVE a separate copy

print(f"PAUSED inside block {PAUSE_LAYER}, after its look-around part")
print("Parts run so far:", parts_run[-3:], f"... ({len(parts_run)} parts in total)")
print("Table entering block:", tuple(saved["block_input"].shape))
print("Look-around result:  ", tuple(saved["look_result"].shape), " (1 sentence, 8 word-pieces, 768 numbers each)")
print("First 5 numbers of the look-around result for ' speak':", saved["look_result"][0, -1, :5].tolist())

PAUSED inside block 8, after its look-around part
Parts run so far: ['look 7', 'think 7', 'look 8'] ... (17 parts in total)
Table entering block: (1, 8, 768)
Look-around result:   (1, 8, 768)  (1 sentence, 8 word-pieces, 768 numbers each)
First 5 numbers of the look-around result for ' speak': [-1.4078274965286255, -0.7360004186630249, -0.5683465600013733, -1.2349365949630737, 0.35659319162368774]


## Step 7: Show what ' speak' was looking at in this block
The look-around part has **12 sub-parts (called heads)**, each looking at the earlier words in its own way.
For the last word ' speak', we show how much attention (out of 100%) it gives to each word,
averaged over the 12 heads, and for the single head that looks most at ' Paris'.

To be sure this display shows what the model really used, we rebuild the look-around result from these numbers
and check it matches the saved one.

In [8]:
def look_pattern(i, block_input):
    """How much each word looks at each earlier word in block i, for all 12 heads."""
    attn = gpt2.h[i].attn
    n, heads, size = block_input.shape[1], attn.num_heads, attn.head_dim
    q, k, v = attn.c_attn(gpt2.h[i].ln_1(block_input)).split(attn.embed_dim, dim=2)
    q, k, v = (x.view(1, n, heads, size).transpose(1, 2) for x in (q, k, v))
    scores = (q @ k.transpose(-1, -2)) * attn.scaling
    future = torch.triu(torch.ones(n, n, dtype=torch.bool, device=scores.device), 1)
    weights = scores.masked_fill(future, float("-inf")).softmax(-1)   # (1, heads, n, n)
    rebuilt = attn.c_proj((weights @ v).transpose(1, 2).reshape(1, n, attn.embed_dim))
    return weights, rebuilt

weights, rebuilt = look_pattern(PAUSE_LAYER, saved["block_input"])
print("Rebuilt result matches saved look-around result:",
      torch.allclose(rebuilt, saved["look_result"], atol=1e-4))

PARIS_POS = 3
from_speak = weights[0, :, -1]                     # 12 heads x 8 words
best_head = from_speak[:, PARIS_POS].argmax().item()
print(f"\nWhere ' speak' looks in block {PAUSE_LAYER}:")
print(f"  {'word':<10}{'average of 12 heads':>21}{f'head {best_head}':>10}")
for w, avg, best in zip(words, from_speak.mean(0).tolist(), from_speak[best_head].tolist()):
    print(f"  {w!r:<10}{avg:>21.0%}{best:>10.0%}")

Rebuilt result matches saved look-around result: True

Where ' speak' looks in block 8:
  word        average of 12 heads   head 11
  'I'                         72%       40%
  ' live'                      1%        1%
  ' in'                        1%        1%
  ' Paris'                    11%       49%
  ','                          5%        6%
  ' so'                        4%        1%
  ' I'                         4%        0%
  ' speak'                     2%        1%


## Step 8: RESUME with the saved result unchanged (check)
Only the think part of block 8 and blocks 9 to 11 should run, and the answer should be exactly the normal one.

In [9]:
parts_run.clear()
resumed_logits = resume_after_look({k: (v.clone() if torch.is_tensor(v) else v) for k, v in saved.items()})  # <-- RESUME

print("RESUMED. Parts run:", parts_run)
print("Largest difference from normal model:", (resumed_logits - normal_logits).abs().max().item())
expected = [f"think {PAUSE_LAYER}"] + [f"{p} {i}" for i in range(PAUSE_LAYER + 1, NUM_BLOCKS) for p in ("look", "think")]
assert parts_run == expected, "Something earlier was rerun!"
assert torch.allclose(resumed_logits, normal_logits, atol=1e-4)
print("OK: resuming gives the normal answer, and nothing before the pause was rerun")

RESUMED. Parts run: ['think 8', 'look 9', 'think 9', 'look 10', 'think 10', 'look 11', 'think 11']


Largest difference from normal model: 0.0
OK: resuming gives the normal answer, and nothing before the pause was rerun


## Step 9: MODIFY (a) — switch off the look-around result for ' speak'
We set the look-around result of the last word to zero, so in block 8 ' speak' gains nothing from the other words.
All other words are left unchanged.

In [10]:
def switch_off_last(look_result):
    look_result = look_result.clone()
    look_result[0, -1] = 0
    return look_result

ckpt_a = {**saved, "look_result": switch_off_last(saved["look_result"])}  # <-- MODIFY

parts_run.clear()
logits_a = resume_after_look(ckpt_a)  # <-- RESUME
print("RESUMED. Parts run:", parts_run, "\n")
compare(logits_a, "Switched off (block 8)")

RESUMED. Parts run: ['think 8', 'look 9', 'think 9', 'look 10', 'think 10', 'look 11', 'think 11'] 

Normal model              Switched off (block 8)
  ' French'    63.05%      ' French'    51.22%
  ' English'    4.93%      ' English'    4.34%
  ' to'         3.16%      ' to'         3.92%
  ' a'          2.94%      ' a'          3.14%
  ' with'       1.76%      ' with'       2.46%

  ' French': 63.05% -> 51.22%
  ' German':  0.62% ->  0.69%


## Step 10: MODIFY (b) — replace it with the result from the Berlin sentence
We pause the sentence *"I live in Berlin, so I speak"* at the same point, and copy its look-around result for ' speak'
into our checkpoint. Only the row for ' speak' changes.

In [11]:
OTHER_PROMPT = "I live in Berlin, so I speak"
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape

other_ckpt = run_until_look(other_ids, PAUSE_LAYER)

def swap_last(look_result, other_look_result):
    look_result = look_result.clone()
    look_result[0, -1] = other_look_result[0, -1]
    return look_result

ckpt_b = {**saved, "look_result": swap_last(saved["look_result"], other_ckpt["look_result"])}  # <-- MODIFY

parts_run.clear()
logits_b = resume_after_look(ckpt_b)  # <-- RESUME
print("RESUMED. Parts run:", parts_run, "\n")
compare(logits_b, "Berlin version (block 8)")

RESUMED. Parts run: ['think 8', 'look 9', 'think 9', 'look 10', 'think 10', 'look 11', 'think 11'] 

Normal model              Berlin version (block 8)
  ' French'    63.05%      ' French'    53.61%
  ' English'    4.93%      ' English'    7.04%
  ' to'         3.16%      ' to'         4.44%
  ' a'          2.94%      ' German'     3.33%
  ' with'       1.76%      ' a'          2.78%

  ' French': 63.05% -> 53.61%
  ' German':  0.62% ->  3.33%


## Step 11: Try the same two changes in every block, one block at a time

In [12]:
other_look = {i: run_until_look(other_ids, i)["look_result"] for i in range(NUM_BLOCKS)}

print(f"{'Block':<7}{'Switched off':^30}{'Berlin version':^30}")
print(f"{'':<7}{'top word':<12}{'French':>8}{'German':>8}  {'top word':<12}{'French':>8}{'German':>8}")
for layer in range(NUM_BLOCKS):
    ckpt = run_until_look(input_ids, layer)
    row = f"{layer:<7}"
    for new_look in (switch_off_last(ckpt["look_result"]), swap_last(ckpt["look_result"], other_look[layer])):
        p = resume_after_look({**ckpt, "look_result": new_look})[0, -1].softmax(-1)
        row += f"{tokenizer.decode(p.argmax())!r:<12}{p[FRENCH].item():>8.1%}{p[GERMAN].item():>8.1%}  "
    print(row)

Block           Switched off                 Berlin version        
       top word      French  German  top word      French  German


0      ' of'           0.0%    0.0%  ' French'      63.5%    0.7%  


1      ' French'      63.9%    0.6%  ' French'      63.7%    0.6%  
2      ' French'      60.4%    0.7%  ' French'      65.1%    0.7%  


3      ' French'      56.1%    0.5%  ' French'      63.5%    0.6%  


4      ' French'      60.7%    0.6%  ' French'      64.1%    0.6%  


5      ' French'      65.2%    0.5%  ' French'      63.3%    0.6%  
6      ' French'      63.4%    0.5%  ' French'      63.1%    0.6%  


7      ' French'      60.4%    0.6%  ' French'      63.5%    0.6%  
8      ' French'      51.2%    0.7%  ' French'      53.6%    3.3%  


9      ' French'      70.2%    0.6%  ' French'      62.9%    1.9%  


10     ' French'      74.5%    0.5%  ' French'      62.4%    1.8%  
11     ' to'          11.5%    0.7%  ' French'      33.1%    6.8%  


## Step 12: Change blocks 8 to 11 together
One block on its own has only a small effect, so here we pause after the look-around part of **each** of
blocks 8, 9, 10 and 11 in the same run, and change the result for ' speak' every time.

In [13]:
LAYERS = [8, 9, 10, 11]

parts_run.clear()
logits_off = run_model(input_ids, edits={i: switch_off_last for i in LAYERS})
assert len(parts_run) == 2 * NUM_BLOCKS, "Each part should run exactly once"
compare(logits_off, "Switched off (8-11)")

print()
logits_swap = run_model(input_ids, edits={i: (lambda r, i=i: swap_last(r, other_look[i])) for i in LAYERS})
compare(logits_swap, "Berlin version (8-11)")

Normal model              Switched off (8-11)
  ' French'    63.05%      ' to'        15.70%
  ' English'    4.93%      ' the'        9.22%
  ' to'         3.16%      ' for'        5.14%
  ' a'          2.94%      ' a'          4.04%
  ' with'       1.76%      ' English'    3.98%

  ' French': 63.05% ->  1.83%
  ' German':  0.62% ->  1.38%

Normal model              Berlin version (8-11)
  ' French'    63.05%      ' German'    57.11%
  ' English'    4.93%      ' to'         6.04%
  ' to'         3.16%      ' English'    4.56%
  ' a'          2.94%      ' a'          3.72%
  ' with'       1.76%      ' the'        2.21%

  ' French': 63.05% ->  0.85%
  ' German':  0.62% -> 57.11%
